# v2a-RSN stability metrics (c-GC*)

This notebook loads each c-GC* adjacency `.pkl` file in `outputs/v2a-RSNs/c-GC-star/`, normalizes the adjacency matrices for `P = 1..7`, and computes the observed ground-truth-free stability metrics used by the v2a-RSN analysis:

- edge counts by conditioning depth
- `D_p`
- `D_minus` and `D_plus`
- `T_obs` and the depth where it occurs
- first-step edge loss, drop to the minimum edge count, final edge loss, cumulative instability, and deletion/addition counts

These are observed descriptive metrics. They do not by themselves provide surrogate-null bootstrap bands, calibrated p-values, type-I error, or power. They are the observed statistics that a future empirical calibration step would compare against surrogate-null distributions.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find '{CAUSALISED_GC_RELATIVE_PATH}' from {Path.cwd().resolve()}"
    )

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from markovianity_diagnostic.experiments.graph_metrics import compute_graph_stability_metrics
from markovianity_diagnostic.experiments.adapters import analyze_with_gcstar_cgc
from markovianity_diagnostic.experiments.v2a_rsn_utils import (
    load_and_filter_traces,
    append_multiple_csvs,
    setup_recording_paths,
)

print(f'Project root: {PROJECT_ROOT}')


## Configuration

Set the recording name below. This notebook processes a single recording with checkpoint/resumption support.

In [ ]:
# CONFIGURATION: Specify which recording to process
RECORDING_NAME = '220210_F2_run5'  # This notebook processes 220210_F2_run5

METHOD_DIR = 'c-GC-star'

# Available recordings:
# - 220119_F2_run11
# - 220127_F4_F4_run2  
# - 220210_F1_F1_run6
# - 220210_F2_F2_run5


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find '{CAUSALISED_GC_RELATIVE_PATH}' from {Path.cwd().resolve()}"
    )

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from markovianity_diagnostic.experiments.graph_metrics import compute_graph_stability_metrics
from markovianity_diagnostic.experiments.v2a_rsn_utils import (
    load_checkpoint,
    save_checkpoint,
    log_processing_run,
    append_multiple_csvs,
    load_and_filter_adjacencies,
    setup_recording_paths,
    find_pkl_file,
)

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
summary_columns = [
    'dataset',
    'file',
    'n_nodes',
    'n_possible_edges',
    'T_obs',
    'T_obs_depth',
    'T_minus_obs',
    'T_plus_obs',
    'first_step_drop_count',
    'first_step_drop_fraction',
    'drop_to_min_count',
    'drop_to_min_fraction',
    'final_drop_count',
    'final_drop_fraction',
    'cumulative_D_p',
    'cumulative_D_minus',
    'cumulative_D_plus',
    'mean_post_first_D_p',
    'max_post_first_D_p',
]
summary_df[summary_columns]


In [ ]:
transition_columns = [
    'dataset',
    'P',
    'edge_count',
    'previous_edge_count',
    'edge_delta',
    'edge_deletions',
    'edge_additions',
    'net_deletion_count',
    'D_p',
    'D_minus',
    'D_plus',
    'deletion_share_of_changes',
    'addition_share_of_changes',
    'net_deletion_fraction_possible',
]
transition_df[transition_columns]


## Saved outputs

The notebook writes the observed stability metrics to the method output directory:

- `summary.csv`: one row per recording, including `T_obs`, first-step drop, drop to minimum, final drop, and cumulative instability
- `transitions.csv`: one row per recording and conditioning-depth transition, including `D_p`, `D_minus`, `D_plus`, deletions, additions, and net edge loss
- `diagnostic_summary.csv`: duplicate of `summary.csv` with the expanded diagnostic schema, kept for explicit downstream use
- `transition_diagnostics.csv`: duplicate of `transitions.csv` with the expanded diagnostic schema, kept for explicit downstream use
- `summary.json`: nested JSON version of the summary metrics

These files still contain observed statistics only. Calibrated bootstrap bands and p-values require rerunning the learner on surrogate time series generated under an order-`p0` null.


In [ ]:
# Plot edge counts vs n_pasts for each dataset.
import matplotlib.pyplot as plt

plot_df = pd.read_csv(OUTPUT_DIR / 'transitions.csv')
pivot = plot_df.pivot(index='P', columns='dataset', values='edge_count').sort_index()

# Map datasets to fish-1..4 labels in order.
datasets = list(pivot.columns)
labels = {datasets[i]: f'fish-{i + 1}' for i in range(len(datasets))}
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

fig, ax = plt.subplots(figsize=(9, 6))
for i, dataset in enumerate(datasets):
    ax.plot(pivot.index, pivot[dataset], marker='o', color=colors[i], label=labels[dataset])

ax.set_xlabel('$n_{pasts}$')
ax.set_ylabel('$n_{links}$', rotation=45)
ax.set_title('$n_{pasts}$ against edge counts')
ax.set_xticks(pivot.index.tolist())
ax.legend()
ax.grid(False)
plt.tight_layout()

out_path = OUTPUT_DIR / 'npasts_links.png'
fig.savefig(out_path, dpi=200)
print(f'Saved plot to: {out_path}')
plt.show()


In [ ]:
# Plot observed D_p decomposition. These are not bootstrap bands.
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
axes = axes.ravel()

datasets = sorted(transition_df['dataset'].unique())
recording_labels = {dataset: f'fish-{i + 1}' for i, dataset in enumerate(datasets)}

for ax, dataset in zip(axes, datasets):
    subset = transition_df[(transition_df['dataset'] == dataset) & transition_df['D_p'].notna()]
    ax.plot(subset['P'], subset['D_p'], marker='o', label='$D_p$')
    ax.plot(subset['P'], subset['D_minus'], marker='v', label='$D_p^-$')
    ax.plot(subset['P'], subset['D_plus'], marker='^', label='$D_p^+$')
    ax.set_title(recording_labels[dataset])
    ax.set_xlabel('$n_{pasts}$')
    ax.set_ylabel('normalized instability')
    ax.grid(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3)
fig.tight_layout(rect=(0, 0, 1, 0.94))

out_path = OUTPUT_DIR / 'instability_decomposition.png'
fig.savefig(out_path, dpi=200)
print(f'Saved plot to: {out_path}')
plt.show()


In [ ]:
# Setup paths
paths = setup_recording_paths(PROJECT_ROOT, RECORDING_NAME, METHOD_DIR)
recording_dir = paths['recording_dir']
output_dir = paths['output_dir']
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Recording: {RECORDING_NAME}')
print(f'Output dir: {output_dir}')
print()

# Load and filter traces (neurons × frames)
X = load_and_filter_traces(recording_dir, RECORDING_NAME)
print(f'Filtered traces: {X.shape}')
print()

# Run c-GC analysis on each p_value with progress bar
P_VALUES = [1, 2, 3, 4, 5, 6, 7]
print('Running c-GC analysis...')
adjacencies = {}
for p_value in tqdm(P_VALUES, desc='Computing p_values', unit='p'):
    result = analyze_with_gcstar_cgc(X, [p_value])
    adjacencies.update(result)
print(f'Computed adjacencies for p_values: {sorted(adjacencies.keys())}')
print()

# Save adjacency dict as pickle
connectivity_pkl = output_dir / 'connectivity.pkl'
pd.to_pickle(adjacencies, connectivity_pkl)
print(f'Saved adjacencies to: {connectivity_pkl.name}')

# Prepare adjacencies for metric computation (binary, no diagonals)
adjacencies_binary = {p: (adjacencies[p] != 0).astype(int) for p in adjacencies}
for p in adjacencies_binary:
    np.fill_diagonal(adjacencies_binary[p], 0)

# Compute stability metrics
metrics = compute_graph_stability_metrics(adjacencies_binary)

# Build summary and transition tables
summary = build_summary(RECORDING_NAME, connectivity_pkl, adjacencies_binary, metrics)
transition_df = build_transition_df(RECORDING_NAME, connectivity_pkl.name, adjacencies_binary, metrics)
summary_df = pd.DataFrame([summary])

# Prepare dataframe types
for col in ['previous_P', 'previous_edge_count', 'edge_delta', 'net_edge_loss_count',
            'edge_deletions', 'edge_additions', 'net_deletion_count']:
    transition_df[col] = transition_df[col].astype('Int64')

# Append results to CSV files
append_multiple_csvs(summary_df, transition_df, output_dir)

# Save JSON summary
summary_json = {k: v for k, v in summary.items() if k not in ['edge_counts', 'D_p', 'D_parts']}
summary_json.update({k: summary[k] for k in ['edge_counts', 'D_p', 'D_parts']})
with open(output_dir / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_json, f, indent=2)

print()
print(f'✓ Loaded and filtered traces: {X.shape}')
print(f'✓ Ran c-GC analysis')
print(f'✓ Saved adjacencies: {connectivity_pkl.name}')
print(f'✓ Computed stability metrics')
print(f'✓ Saved outputs to: {output_dir.name}/')


In [ ]:
def _safe_fraction(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else float('nan')


def _argmax_key(values: dict[int, float]) -> int | None:
    return max(values, key=values.get) if values else None


def build_summary(
    dataset_name: str,
    path: Path,
    adjacencies: dict[int, np.ndarray],
    metrics: dict,
) -> dict:
    """Build summary metrics dictionary."""
    EXPECTED_P_VALUES = list(range(1, 8))
    n_nodes = int(next(iter(adjacencies.values())).shape[0])
    n_possible_edges = n_nodes * (n_nodes - 1)
    edge_counts = {int(k): int(v) for k, v in metrics['edge_counts'].items()}
    d_p = {int(k): float(v) for k, v in metrics['D_p'].items()}
    d_parts = {
        int(k): {'D_minus': float(v['D_minus']), 'D_plus': float(v['D_plus'])}
        for k, v in metrics['D_parts'].items()
    }

    min_edge_depth = min(edge_counts, key=edge_counts.get)
    min_edge_count = edge_counts[min_edge_depth]
    final_depth = max(edge_counts)
    first_depth = min(edge_counts)
    first_transition_depth = EXPECTED_P_VALUES[1]
    t_obs_depth = _argmax_key(d_p)
    t_minus_depth = _argmax_key({p: parts['D_minus'] for p, parts in d_parts.items()})
    t_plus_depth = _argmax_key({p: parts['D_plus'] for p, parts in d_parts.items()})

    first_edge_count = edge_counts[first_depth]
    first_step_drop_count = edge_counts[first_depth] - edge_counts[first_transition_depth]
    drop_to_min_count = edge_counts[first_depth] - min_edge_count
    final_drop_count = edge_counts[first_depth] - edge_counts[final_depth]

    post_first_d = {p: value for p, value in d_p.items() if p > first_transition_depth}
    return {
        'dataset': dataset_name,
        'file': path.name,
        'n_nodes': n_nodes,
        'n_possible_edges': n_possible_edges,
        'n_p_values': len(adjacencies),
        'T_obs': float(metrics['T_obs']),
        'T_obs_depth': int(t_obs_depth) if t_obs_depth is not None else None,
        'T_minus_obs': float(d_parts[t_minus_depth]['D_minus']) if t_minus_depth is not None else 0.0,
        'T_minus_depth': int(t_minus_depth) if t_minus_depth is not None else None,
        'T_plus_obs': float(d_parts[t_plus_depth]['D_plus']) if t_plus_depth is not None else 0.0,
        'T_plus_depth': int(t_plus_depth) if t_plus_depth is not None else None,
        'edge_count_p1': int(edge_counts[first_depth]),
        'edge_count_pmax': int(edge_counts[final_depth]),
        'min_edge_count': int(min_edge_count),
        'min_edge_depth': int(min_edge_depth),
        'first_step_drop_count': int(first_step_drop_count),
        'first_step_drop_fraction': _safe_fraction(first_step_drop_count, first_edge_count),
        'drop_to_min_count': int(drop_to_min_count),
        'drop_to_min_fraction': _safe_fraction(drop_to_min_count, first_edge_count),
        'final_drop_count': int(final_drop_count),
        'final_drop_fraction': _safe_fraction(final_drop_count, first_edge_count),
        'cumulative_D_p': float(sum(d_p.values())),
        'cumulative_D_minus': float(sum(parts['D_minus'] for parts in d_parts.values())),
        'cumulative_D_plus': float(sum(parts['D_plus'] for parts in d_parts.values())),
        'mean_post_first_D_p': float(np.mean(list(post_first_d.values()))) if post_first_d else float('nan'),
        'max_post_first_D_p': float(max(post_first_d.values())) if post_first_d else float('nan'),
        'edge_counts': edge_counts,
        'D_p': d_p,
        'D_parts': d_parts,
    }


def build_transition_df(
    dataset_name: str,
    file_name: str,
    adjacencies: dict[int, np.ndarray],
    metrics: dict,
) -> pd.DataFrame:
    """Build transition metrics dataframe."""
    EXPECTED_P_VALUES = list(range(1, 8))
    n_nodes = int(next(iter(adjacencies.values())).shape[0])
    n_possible_edges = n_nodes * (n_nodes - 1)
    edge_counts = {int(k): int(v) for k, v in metrics['edge_counts'].items()}
    d_p = {int(k): float(v) for k, v in metrics['D_p'].items()}
    d_parts = {
        int(k): {'D_minus': float(v['D_minus']), 'D_plus': float(v['D_plus'])}
        for k, v in metrics['D_parts'].items()
    }

    rows = []
    for p_value in EXPECTED_P_VALUES:
        row = {'dataset': dataset_name, 'file': file_name, 'P': p_value, 'edge_count': int(edge_counts[p_value])}
        if p_value in d_p:
            previous_p = EXPECTED_P_VALUES[EXPECTED_P_VALUES.index(p_value) - 1]
            previous_count = edge_counts[previous_p]
            d_minus = d_parts[p_value]['D_minus']
            d_plus = d_parts[p_value]['D_plus']
            d_value = d_p[p_value]
            deletion_count = int(round(d_minus * n_possible_edges))
            addition_count = int(round(d_plus * n_possible_edges))
            edge_delta = edge_counts[p_value] - previous_count
            row.update({
                'previous_P': previous_p,
                'previous_edge_count': int(previous_count),
                'edge_delta': int(edge_delta),
                'net_edge_loss_count': int(-edge_delta),
                'D_p': float(d_value),
                'D_minus': float(d_minus),
                'D_plus': float(d_plus),
                'edge_deletions': deletion_count,
                'edge_additions': addition_count,
                'net_deletion_count': int(deletion_count - addition_count),
                'deletion_share_of_changes': _safe_fraction(d_minus, d_value),
                'addition_share_of_changes': _safe_fraction(d_plus, d_value),
                'net_deletion_fraction_possible': float(d_minus - d_plus),
                'edge_delta_fraction_previous': _safe_fraction(edge_delta, previous_count),
            })
        else:
            row.update({k: np.nan for k in [
                'previous_P', 'previous_edge_count', 'edge_delta', 'net_edge_loss_count',
                'D_p', 'D_minus', 'D_plus', 'edge_deletions', 'edge_additions',
                'net_deletion_count', 'deletion_share_of_changes', 'addition_share_of_changes',
                'net_deletion_fraction_possible', 'edge_delta_fraction_previous'
            ]})
        rows.append(row)
    return pd.DataFrame(rows)
